# **Start**

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
os.environ["PIP_CONSTRAINT"] = "/tmp/numpy_constraint.txt"
!echo "numpy==1.26.4" > /tmp/numpy_constraint.txt

!pip uninstall -y torch torchaudio torchvision \
    torchao torchcodec torchdata torchtune torchsummary -q 2>/dev/null

!pip uninstall -y tensorflow tensorflow-text tensorflow-hub tf-keras \
    tensorflow_decision_forests tensorflow-probability \
    tensorflow-Datasets tensorflow-metadata -q 2>/dev/null

!pip uninstall -y numpy scikit-learn shap xgboost lightgbm dask \
    seaborn plotly openpyxl Cython catboost interpret lime -q 2>/dev/null

!pip install numpy==1.26.4 -q
!pip install scikit-learn==1.6.1 -q
!pip install torch==2.9.0 -q

!pip install lightgbm==4.6.0 -q
!pip install xgboost==3.1.2 -q
!pip install catboost==1.2.8 -q
!pip install gpboost==1.6.1 -q
!pip install ngboost==0.5.8 -q
!pip install pgbm==2.2.0 -q
!pip install pytorch-tabnet2==4.5.3 -q

!pip install bayesian-optimization==3.2.0 -q
!pip install optuna==4.6.0 -q
!pip install optunahub==0.4.0 -q
!pip install cmaes==0.12.0 -q

!pip install shap==0.44.0 -q
!pip install lime==0.2.0.1 -q
!pip install interpret==0.7.4 -q

!pip install mapie==0.6.0 -q
!pip install puncc==0.8.0 -q
!pip install skorch==1.3.1 -q
!pip install properscoring==0.1 -q

!pip install dask[dataframe]==2025.12.0 -q
!pip install cython==3.0.12 -q
!pip install seaborn==0.13.2 -q
!pip install plotly==5.24.1 -q
!pip install kaleido==1.2.0 -q
!pip install openpyxl==3.1.5 -q
!pip install XlsxWriter==3.2.9 -q
!pip install cp==2020.12.3 -q

!pip install numpy==1.26.4 --force-reinstall --no-deps -q

os._exit(0)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 87.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
spopt 0.7.0 requires scikit-learn>=1.4.0, which is not installed.
accelerate 1.13.0 requires torch>=2.0.0, which is not installed.
cufflinks 0.17.3 requires plotly>=4.1.1, which is not installed.
peft 0.19.1 requires torch>=1.13.0, which is not installed.
spreg 1.9.0 requires scikit-learn>=0.22, which is not installed.
sentence-transformers 5.4.1 requires scikit-learn>=0.22.0, which is not installed.
sentence-transformers 5.4.1 requires torch>=1.11.0, which is not installed.
fastai 2.8.7 requires scikit-learn, which is not installed.
fastai 2.8.7 requires torch<3,>=1.10, which is not installed.
fastai 2.8.7 requires torchvision>=0.11, which is not installed.

# **Imports**

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ngboost
import gpboost
from scipy.stats import randint
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.svm import SVR
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from gpboost import GPBoostRegressor
from ngboost import NGBRegressor
import optuna
import optunahub
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import HalvingGridSearchCV
from interpret import show
from interpret.blackbox import LimeTabular, ShapKernel
from optuna.samplers import RandomSampler
import random
import time
from ngboost.distns import Normal
from ngboost.scores import LogScore
from scipy.stats import norm
from interpret import set_visualize_provider
from interpret.provider import InlineProvider
from interpret.glassbox import ExplainableBoostingRegressor
from interpret import show
import plotly.express as px
from io import BytesIO
from openpyxl import Workbook, load_workbook
import os
from openpyxl.drawing.image import Image as openpyxlImage
import warnings
import xlsxwriter
from openpyxl.drawing.image import Image
from pgbm.sklearn import HistGradientBoostingRegressor
import torch
from pgbm.torch import PGBM
import plotly.graph_objects as go
warnings.filterwarnings('ignore')
import pickle
import json
from pytorch_tabnet import TabNetRegressor

In [2]:
# Go to find & replace button and replace (pile_settlement_uncertainty_analysis) with your folder name. Rename your train and test Dataset as train.csv and test.csv.
# Modify the names of the feature in the below cell.
# Replace (str) with actual data label name.

In [3]:
feature_names = ['D', 'L', 'N30_Ava', 'Cu_Ava', 'N30_Base', 'Cu_Base', 'UL', 'Q ']

In [4]:
train_data_path = "./drive/MyDrive/Regression/pile_settlement_uncertainty_analysis/data/train.csv"
test_data_path = "./drive/MyDrive/Regression/pile_settlement_uncertainty_analysis/data/test.csv"
train_data = pd.read_csv(train_data_path)
test_data = pd.read_csv(test_data_path)
print("Training data loaded successfully.")
print("Test data loaded successfully.")

Training data loaded successfully.
Test data loaded successfully.


In [5]:
print("\nShape of training data:", train_data.shape)
print("First 5 rows of training data:\n", train_data.head(5))
print("\nShape of test data:", test_data.shape)
print("First 5 rows of test data:\n", test_data.head(5))


Shape of training data: (189, 8)
First 5 rows of training data:
         C    mp     FA      CA       F       W_P    Adm    str
0  280.80  70.2  858.0  1183.0    0.00  0.450000  0.610  21.56
1  372.15   0.0  975.0   525.0   52.85  0.493081  7.000  34.00
2  360.00  75.0  975.0   525.0   40.00  0.509722  7.000  28.00
3  364.30   0.0  975.0   525.0  110.40  0.503706  7.000  42.00
4  315.00  31.5  780.0  1110.0    0.00  0.370000  5.355  25.82

Shape of test data: (95, 8)
First 5 rows of test data:
         C     mp     FA     CA      F       W_P   Adm   str
0  364.30    0.0  975.0  525.0  60.40  0.503706   7.0  34.0
1  344.30   75.0  975.0  525.0  55.40  0.532965   7.0  36.0
2  390.00    0.0  975.0  525.0  60.00  0.470513   7.0  36.0
3  352.15   75.0  975.0  525.0  47.85  0.521085   7.0  35.0
4  400.00  160.0  801.0  801.0  40.00  0.300000  10.3  44.6


In [6]:
X_train = train_data.iloc[:, :-1]
y_train = train_data.iloc[:, -1]
X_test = test_data.iloc[:, :-1]
y_test = test_data.iloc[:, -1]
x_test= X_test
print("\nShape of X_train:", X_train.shape)
print("Shape of y_train:", y_train.shape)
print("Shape of X_test:", X_test.shape)
print("Shape of y_test:", y_test.shape)


Shape of X_train: (189, 7)
Shape of y_train: (189,)
Shape of X_test: (95, 7)
Shape of y_test: (95,)


In [7]:
# Apply z-score normalization
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Print the first five rows of the normalized data
print("\nFirst five rows of normalized X_train:")
print(X_train[:5])

print("\nFirst five rows of normalized X_test:")
print(X_test[:5])


First five rows of normalized X_train:
[[-1.56879184  0.95792014 -0.09685034  1.33138508 -0.85369002  0.16810471
  -0.08799752]
 [ 0.40300047 -0.7708921   0.73537613 -0.93459372  0.061075    0.78285857
  -0.0664621 ]
 [ 0.14074238  1.07612953  0.73537613 -0.93459372 -0.16134185  1.020321
  -0.0664621 ]
 [ 0.233558   -0.7708921   0.73537613 -0.93459372  1.05719093  0.93447436
  -0.0664621 ]
 [-0.83058388  0.00485698 -0.65166799  1.07999229 -0.85369002 -0.97347298
  -0.07200604]]

First five rows of normalized X_test:
[[ 0.233558   -0.7708921   0.73537613 -0.93459372  0.19175572  0.93447436
  -0.0664621 ]
 [-0.19814256  1.07612953  0.73537613 -0.93459372  0.1052122   1.35199213
  -0.0664621 ]
 [ 0.78829322 -0.7708921   0.73537613 -0.93459372  0.18483223  0.4608195
  -0.0664621 ]
 [-0.02870009  1.07612953  0.73537613 -0.93459372 -0.02546852  1.18246784
  -0.0664621 ]
 [ 1.0041435   3.1694207  -0.50229401  0.01587763 -0.16134185 -1.97235346
  -0.05534052]]


# **Functions**

In [9]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import os
def plot_best_scores(best_scores_ran, excel_file_path):
    # Extract the best pruner for each model based on RMSE and correlation coefficient
    best_rmse_scores = {}
    best_corr_coef_scores = {}

    for (model_name, pruner_name), scores in best_scores_ran.items():
        # Initialize if not already present
        if model_name not in best_rmse_scores:
            best_rmse_scores[model_name] = (scores['test_rmse'], pruner_name)
        if model_name not in best_corr_coef_scores:
            best_corr_coef_scores[model_name] = (scores['test_corr_coef'], pruner_name)

        # Update if better scores are found
        if scores['test_rmse'] < best_rmse_scores[model_name][0]:
            best_rmse_scores[model_name] = (scores['test_rmse'], pruner_name)
        if scores['test_corr_coef'] > best_corr_coef_scores[model_name][0]:
            best_corr_coef_scores[model_name] = (scores['test_corr_coef'], pruner_name)

    # Prepare data for plotting
    model_names_rmse = [f"{model} ({pruner})" for model, (rmse, pruner) in best_rmse_scores.items()]
    rmse_values = [rmse for rmse, _ in best_rmse_scores.values()]

    model_names_corr = [f"{model} ({pruner})" for model, (corr, pruner) in best_corr_coef_scores.items()]
    corr_values = [corr for corr, _ in best_corr_coef_scores.values()]

    # Plot RMSE
    plt.figure(figsize=(12, 6))
    bars_rmse = plt.bar(model_names_rmse, rmse_values, color='skyblue')

    # Highlight the best model
    best_rmse_index = np.argmin(rmse_values)
    bars_rmse[best_rmse_index].set_color('orange')

    # Annotate the bars with the RMSE scores
    for i, bar in enumerate(bars_rmse):
        plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() - 0.05,
                 f'{rmse_values[i]:.5f}', ha='center', va='bottom', color='black')

    # Add labels and title for the RMSE bar chart
    plt.xlabel('Model (Pruner)')
    plt.ylabel('Test RMSE')
    plt.title('Best Test RMSE for Each Model')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    rmse_image_path = 'rmse_plot.png'
    _ensure_parent_dir(rmse_image_path)
    plt.savefig(_ensure_parent_dir(rmse_image_path))
    plt.close()

    # Plot Correlation Coefficient
    plt.figure(figsize=(12, 6))
    bars_corr = plt.bar(model_names_corr, corr_values, color='lightgreen')

    # Highlight the best model
    best_corr_index = np.argmax(corr_values)
    bars_corr[best_corr_index].set_color('orange')

    # Annotate the bars with the correlation coefficient scores
    for i, bar in enumerate(bars_corr):
        plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() - 0.05,
                 f'{corr_values[i]:.5f}', ha='center', va='bottom', color='black')

    # Add labels and title for the correlation coefficient bar chart
    plt.xlabel('Model (Pruner)')
    plt.ylabel('Correlation Coefficient')
    plt.title('Best Correlation Coefficient for Each Model')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    corr_image_path = 'corr_plot.png'
    _ensure_parent_dir(corr_image_path)
    plt.savefig(_ensure_parent_dir(corr_image_path))
    plt.close()

    # Load the existing Excel file
    workbook = load_workbook(_ensure_excel_file(excel_file_path))

    # Create a new sheet for the plots
    sheet_name = 'Best Models Plots'
    if sheet_name in workbook.sheetnames:
        sheet = workbook[sheet_name]
    else:
        sheet = workbook.create_sheet(title=sheet_name)

    # Insert images into the new Excel sheet
    img_rmse = Image(rmse_image_path)
    img_corr = Image(corr_image_path)

    # Insert images
    sheet.add_image(img_rmse, 'A1')
    sheet.add_image(img_corr, 'A20')  # Adjust the position as needed

    # Save the workbook
    _ensure_parent_dir(excel_file_path)
    workbook.save(_ensure_parent_dir(excel_file_path))

    # Clean up the image files
    if os.path.exists(str(rmse_image_path)): os.remove(str(rmse_image_path))
    if os.path.exists(str(corr_image_path)): os.remove(str(corr_image_path))

# Example usage
# plot_best_scores(best_scores_ran, 'path_to_your_excel_file.xlsx')

In [10]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import os
def generate_interpretml_explanations_summary_pruners(
    results_dict, X_train, y_train, X_test, feature_names, instance_indices=None, excel_file_path=None
):
    if instance_indices is None:
        instance_indices = range(len(X_test))
    elif isinstance(instance_indices, int):
        instance_indices = [instance_indices]

    valid_indices = [idx for idx in instance_indices if 0 <= idx < len(X_test)]
    if not valid_indices:
        print("No valid instance indices provided.")
        return

    if isinstance(X_test, pd.DataFrame):
        instances_to_explain = X_test.iloc[valid_indices]
    else:
        instances_to_explain = X_test[valid_indices]

    best_model_pruners = {}
    for model_key, model_info in results_dict.items():
        if isinstance(model_key, tuple):
            model_name, pruner_name = model_key
        else:
            model_name = model_key
            pruner_name = None

        best_score = model_info.get('best_score')
        if best_score is None:
            print(f"No 'best_score' found for {model_key}. Skipping this combination.")
            continue

        if model_name not in best_model_pruners:
            best_model_pruners[model_name] = {
                'pruner_name': pruner_name,
                'model_info': model_info,
                'best_score': best_score
            }
        else:
            current_best_score = best_model_pruners[model_name]['best_score']
            if best_score < current_best_score:
                best_model_pruners[model_name] = {
                    'pruner_name': pruner_name,
                    'model_info': model_info,
                    'best_score': best_score
                }

    for model_name, info in best_model_pruners.items():
        pruner_name = info['pruner_name']
        model_info = info['model_info']
        best_params = dict(model_info['best_params'])  # don't mutate original!
        model_class = model_classes.get(model_name)

        if model_class is None:
            print(f"Model {model_name} is not supported or not available.")
            continue

        if model_name == 'CatBoost':
            best_params['verbose'] = 0

        # ------- Main model fit logic ---------
        if model_name == "TabNet":
            # TabNet: reshape y, fit, flatten pred for LIME/SHAP, etc.
            y_train_tabnet = np.array(y_train).reshape(-1, 1)
            try:
                model = model_class(**{k: v for k, v in best_params.items() if k != "verbose"})
            except TypeError:
                model = model_class()
            model.fit(np.array(X_train), y_train_tabnet, max_epochs=100, patience=10, batch_size=1024, eval_set=[(np.array(X_train), y_train_tabnet)])
            def predict_fn(data):
                preds = model.predict(np.array(data))
                # flatten for interpreters
                return preds.flatten()
        else:
            try:
                model = model_class(**best_params)
            except TypeError:
                model = model_class()
            model.fit(X_train, y_train)
            def predict_fn(data):
                return model.predict(data)

        if isinstance(X_train, pd.DataFrame):
            data_for_explainer = X_train.values
        else:
            data_for_explainer = X_train

        if isinstance(instances_to_explain, pd.DataFrame):
            data_for_explanation = instances_to_explain.values
        else:
            data_for_explanation = instances_to_explain

        # Generate LIME explanations
        lime_explainer = LimeTabular(
            predict_fn,
            data=data_for_explainer,
            feature_names=feature_names,
            random_state=1,
            mode='regression'
        )
        lime_explanation = lime_explainer.explain_local(data_for_explanation)

        feature_importances_lime = {}
        num_instances = len(valid_indices)
        for idx in range(num_instances):
            explanation = lime_explanation.data(idx)
            for feature_name, feature_score in zip(explanation['names'], explanation['scores']):
                feature_importances_lime[feature_name] = feature_importances_lime.get(feature_name, 0) + abs(feature_score)
        feature_importances_lime = {k: v / num_instances for k, v in feature_importances_lime.items()}
        feature_importances_lime = {k: round(v, 3) for k, v in feature_importances_lime.items()}

        # Generate SHAP explanations using ShapKernel
        try:
            shap_explainer = ShapKernel(predict_fn, data_for_explainer, feature_names=feature_names)
            shap_explanation = shap_explainer.explain_local(data_for_explanation)

            feature_importances_shap = {}
            for idx in range(num_instances):
                explanation = shap_explanation.data(idx)
                for feature_name, feature_score in zip(explanation['names'], explanation['scores']):
                    feature_importances_shap[feature_name] = feature_importances_shap.get(feature_name, 0) + abs(feature_score)

            feature_importances_shap = {k: v / num_instances for k, v in feature_importances_shap.items()}
            feature_importances_shap = {k: round(v, 3) for k, v in feature_importances_shap.items()}
        except Exception as e:
            print(f"Could not compute SHAP values for model {model_name}: {e}")
            feature_importances_shap = {}

        # Plot LIME and SHAP feature importances side by side
        fig, axes = plt.subplots(1, 2, figsize=(34, 36))

        # Plot LIME feature importances
        lime_importances_df = pd.DataFrame.from_dict(
            feature_importances_lime, orient='index', columns=['importance']
        )
        lime_importances_df.sort_values(by='importance', ascending=False, inplace=True)
        lime_importances_df.plot(kind='bar', legend=False, color='skyblue', ax=axes[0])
        axes[0].set_title(f"LIME Feature Importances for {model_name}")
        axes[0].set_ylabel("Average Absolute Importance Score")
        axes[0].set_xlabel("Features")
        axes[0].tick_params(axis='x', rotation=45)

        for p in axes[0].patches:
            height = p.get_height()
            axes[0].annotate(f'{height:.3f}',
                             (p.get_x() + p.get_width() / 2., height),
                             ha='center', va='bottom', fontsize=8)

        # Plot SHAP feature importances
        shap_importances_df = pd.DataFrame.from_dict(
            feature_importances_shap, orient='index', columns=['importance']
        )
        shap_importances_df.sort_values(by='importance', ascending=False, inplace=True)
        shap_importances_df.plot(kind='bar', legend=False, color='orange', ax=axes[1])
        axes[1].set_title(f"SHAP Feature Importances for {model_name}")
        axes[1].set_ylabel("Average Absolute SHAP Value")
        axes[1].set_xlabel("Features")
        axes[1].tick_params(axis='x', rotation=45)

        for p in axes[1].patches:
            height = p.get_height()
            axes[1].annotate(f'{height:.3f}',
                             (p.get_x() + p.get_width() / 2., height),
                             ha='center', va='bottom', fontsize=8)

        plt.tight_layout()

        # Save plots as images
        image_path = f'feature_importances_{model_name}.png'
        fig.savefig(_ensure_parent_dir(image_path))
        plt.close(fig)

        # Optionally insert images and scores into an Excel file
        if excel_file_path:
            workbook = load_workbook(_ensure_excel_file(excel_file_path))
            sheet_name = f'{model_name} Explanations'
            if sheet_name in workbook.sheetnames:
                sheet = workbook[sheet_name]
            else:
                sheet = workbook.create_sheet(title=sheet_name)

            # Insert images into the new Excel sheet
            img = Image(image_path)
            sheet.add_image(img, 'A1')

            # Create a new sheet for feature importance scores
            scores_sheet_name = f'{model_name} Scores'
            if scores_sheet_name in workbook.sheetnames:
                scores_sheet = workbook[scores_sheet_name]
            else:
                scores_sheet = workbook.create_sheet(title=scores_sheet_name)

            # Write LIME scores
            scores_sheet.append(['Feature', 'LIME Importance'])
            for feature, importance in feature_importances_lime.items():
                scores_sheet.append([feature, importance])

            # Write SHAP scores if available
            if feature_importances_shap:
                scores_sheet.append(['Feature', 'SHAP Importance'])
                for feature, importance in feature_importances_shap.items():
                    scores_sheet.append([feature, importance])

            # Save the workbook
            _ensure_parent_dir(excel_file_path)
            workbook.save(_ensure_parent_dir(excel_file_path))

            # Clean up the image file
            if os.path.exists(str(image_path)): os.remove(str(image_path))

# **Hyperparameter tuning using Autosampler by Optuna**

In [15]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import os
import joblib

def mseloss_objective(yhat, y, sample_weight=None):
    if not torch.is_tensor(yhat):
        yhat = torch.from_numpy(np.array(yhat)).float()
    if not torch.is_tensor(y):
        y = torch.from_numpy(np.array(y)).float()
    gradient = yhat - y
    hessian = torch.ones_like(yhat)
    return gradient, hessian


def rmseloss_metric(yhat, y, sample_weight=None):
    if not torch.is_tensor(yhat):
        yhat = torch.from_numpy(np.array(yhat)).float()
    if not torch.is_tensor(y):
        y = torch.from_numpy(np.array(y)).float()
    loss = torch.sqrt(torch.mean((yhat - y) ** 2))
    return loss


def hyperparameter_tuning_all(X_train, y_train, X_test, y_test, excel_path):

    X_train = np.array(X_train)
    y_train = np.array(y_train)
    X_test = np.array(X_test)
    y_test = np.array(y_test)

    # ================= NEW =================
    model_save_dir = "./drive/MyDrive/Regression/pile_settlement_uncertainty_analysis/hyperparameter_tuning/models"
    os.makedirs(model_save_dir, exist_ok=True)
    best_rmse_tracker = {}
    # =======================================

    models = {
        'Random Forest': (RandomForestRegressor, {
            'n_estimators': [100, 200, 300, 500, 700],
            'criterion': ['squared_error', 'absolute_error', 'friedman_mse', 'poisson'],
            'max_depth': [None, 10, 20, 30, 40],
            'min_samples_split': [2, 5, 10, 0.01],
            'min_samples_leaf': [1, 3, 5, 0.01],
            'min_weight_fraction_leaf': [0.0, 0.01, 0.1, 0.2],
            'max_features': [1.0, 'sqrt', 'log2', 0.3, 0.5],
            'max_leaf_nodes': [None, 50, 100, 200],
            'min_impurity_decrease': [0.0, 0.01, 0.1, 0.2],
            'n_jobs': [-1],
            'random_state': [42],
            'verbose': [0],
            'warm_start': [False],
            'ccp_alpha': [0.0, 0.001, 0.01, 0.05, 0.1]
        }),
        'Gradient Boosting': (GradientBoostingRegressor, {
            'loss': ['squared_error', 'absolute_error', 'huber', 'quantile'],
            'learning_rate': [0.01, 0.05, 0.1, 0.2],
            'n_estimators': [100, 200, 300, 500, 700],
            'subsample': [1.0, 0.9, 0.7, 0.5],
            'criterion': ['friedman_mse', 'squared_error'],
            'min_samples_split': [2, 5, 10, 0.01],
            'min_samples_leaf': [1, 3, 5, 0.01],
            'min_weight_fraction_leaf': [0.0, 0.01, 0.05, 0.1],
            'max_depth': [3, 5, 7, 10],
            'min_impurity_decrease': [0.0, 0.01, 0.1],
            'init': [None],
            'random_state': [42],
            'max_features': [None, 'sqrt', 'log2', 0.5],
            'alpha': [0.9, 0.5, 0.1],
            'verbose': [0],
            'max_leaf_nodes': [None, 10, 30, 50],
            'warm_start': [False],
            'validation_fraction': [0.1],
            'n_iter_no_change': [None, 10, 20],
            'tol': [1e-4, 1e-3],
            'ccp_alpha': [0.0, 0.001, 0.01]
        }),
        'XGBoost': (XGBRegressor, {
            'n_estimators': [100, 200, 300, 400, 500],
            'learning_rate': [0.01, 0.05, 0.1, 0.15],
            'max_depth': [3, 5, 7],
            'min_child_weight': [1, 3, 5],
            'gamma': [0, 0.1, 0.5, 1],
            'subsample': [0.5, 0.6, 0.7, 0.8, 0.9],
            'colsample_bytree': [0.5, 0.7, 0.9],
            'colsample_bylevel': [0.5, 0.7, 0.9],
            'reg_alpha': [0, 0.01, 0.1, 1],
            'reg_lambda': [0.1, 1, 5, 10],
            'objective': ['reg:squarederror'],
            'random_state': [42],
            'n_jobs': [-1]
        }),
        'LightGBM': (LGBMRegressor, {
            'n_estimators': [100, 200, 300, 400, 500],
            'learning_rate': [0.01, 0.05, 0.1, 0.15],
            'num_leaves': [15, 31, 63],
            'max_depth': [3, 5, 7, -1],
            'min_child_samples': [1, 5, 10, 20],
            'subsample': [0.5, 0.6, 0.7, 0.8, 0.9, 1.0],
            'colsample_bytree': [0.5, 0.7, 0.9, 1.0],
            'reg_alpha': [0, 0.01, 0.1, 1],
            'reg_lambda': [0, 0.1, 1, 10],
            'min_child_weight': [1e-5, 1e-3, 1e-2, 1e-1],
            'bagging_freq': [0, 1, 5],
            'objective': ['regression'],
            'random_state': [42],
            'n_jobs': [-1],
            'verbose': [-1]
        }),
        'GPBoost': (GPBoostRegressor, {
            'n_estimators': [100, 200, 300, 400, 500],
            'learning_rate': [0.01, 0.05, 0.1, 0.15],
            'max_depth': [3, 5, 7, -1],
            'num_leaves': [15, 31, 63],
            'min_child_samples': [1, 5, 10, 20],
            'subsample': [0.5, 0.6, 0.7, 0.8, 0.9, 1.0],
            'colsample_bytree': [0.5, 0.7, 0.9, 1.0],
            'reg_alpha': [0, 0.1, 0.5, 1.0],
            'reg_lambda': [0, 0.1, 0.5, 1.0],
            'min_child_weight': [1e-5, 1e-3, 1e-2, 1e-1],
            'random_state': [42],
            'n_jobs': [-1],
            'verbose': [-1]
        }),
        'CatBoost': (CatBoostRegressor, {
            'iterations': [200, 500, 1000],
            'learning_rate': [0.01, 0.03, 0.05, 0.1],
            'depth': [4, 6, 8, 10],
            'l2_leaf_reg': [1, 3, 5, 7, 9],
            'border_count': [32, 64, 128],
            'min_data_in_leaf': [1, 5, 10, 20],
            'rsm': [0.6, 0.8, 1.0],
            'bagging_temperature': [0, 1, 10],
            'random_seed': [42],
            'verbose': [0]
        }),
        'NGBoost': (NGBRegressor, {
            'n_estimators': [200, 500, 1000],
            'learning_rate': [0.01, 0.03, 0.05, 0.1],
            'natural_gradient': [True, False],
            'minibatch_frac': [0.5, 0.7, 0.9, 1.0],
            'col_sample': [0.5, 0.7, 0.9, 1.0],
            'Dist': [Normal],
            'Score': [LogScore],
            'random_state': [42],
            'verbose': [0]
        }),
        'TabNet': (TabNetRegressor, {
            'n_d': [8, 16, 32, 64],
            'n_a': [8, 16, 32, 64],
            'n_steps': [3, 5, 7, 10],
            'gamma': [1.0, 1.3, 1.5, 2.0],
            'lambda_sparse': [1e-4, 1e-3, 1e-2],
            'optimizer_params': [{'lr': 2e-2}], # Fixed learning rate as recommended
            'mask_type': ['sparsemax', 'entmax'],
            'n_shared': [1, 2, 3],
            'n_independent': [1, 2, 3],
            'scheduler_params': [{"step_size": 10, "gamma": 0.9}],
            'scheduler_fn': [torch.optim.lr_scheduler.StepLR],
            'seed': [42],
            'verbose': [0]
        }),
        'HistGradientBoosting': (HistGradientBoostingRegressor, {
            'learning_rate': [0.01, 0.05, 0.1, 0.15],
            'max_iter': [100, 200, 300, 400, 500],
            'max_depth': [3, 5, 7, None],
            'min_samples_leaf': [5, 10, 20],
            'max_leaf_nodes': [15, 31, 63, None],
            'l2_regularization': [0.0, 0.1, 0.5, 1.0],
            'max_bins': [64, 128, 255],
            'early_stopping': [True, False],
            'validation_fraction': [0.1, 0.2],
            'n_iter_no_change': [5, 10, 15],
            'loss': ['squared_error'],
            'random_state': [42],
            'verbose': [0]
        }),
        'PGBM': (PGBM, {})
    }

    pruners = [
        optuna.pruners.MedianPruner(),
        optuna.pruners.NopPruner(),
        optuna.pruners.PatientPruner(optuna.pruners.MedianPruner(), patience=3),
        optuna.pruners.PercentilePruner(25.0),
        optuna.pruners.SuccessiveHalvingPruner(),
        optuna.pruners.HyperbandPruner(),
        optuna.pruners.ThresholdPruner(lower=0.1),
        optuna.pruners.WilcoxonPruner()
    ]

    best_scores = {}
    predictions_df = pd.DataFrame({'Actual': y_test})
    timing_records = []

    for model_name, (model_class, param_space) in models.items():

        rmse_trial_history = {p.__class__.__name__: [] for p in pruners}

        for pruner in pruners:
            pruner_name = pruner.__class__.__name__
            print(f"Running Optuna for {model_name} with {pruner_name}...")
            start_time = time.time()

            best_rmse_tracker[(model_name, pruner_name)] = np.inf

            sampler = optunahub.load_module("samplers/auto_sampler").AutoSampler()
            study = optuna.create_study(direction='minimize', sampler=sampler, pruner=pruner)

            if model_name == 'PGBM':

                def pgbm_objective(trial):
                    params = {
                            'n_estimators': trial.suggest_categorical('n_estimators', [100, 200, 300, 500]),
                            'learning_rate': trial.suggest_categorical('learning_rate', [0.01, 0.05, 0.1, 0.15]),
                            'max_leaves': trial.suggest_int('max_leaves', 15, 63),
                            'min_split_gain': trial.suggest_categorical('min_split_gain', [0.0, 0.1, 0.5, 1.0]),
                            'reg_lambda': trial.suggest_categorical('reg_lambda', [0.1, 1.0, 5.0, 10.0]),
                            'feature_fraction': trial.suggest_categorical('feature_fraction', [0.5, 0.7, 0.9, 1.0]),
                            'bagging_fraction': trial.suggest_categorical('bagging_fraction', [0.5, 0.7, 0.9, 1.0]),
                            'tree_correlation': trial.suggest_categorical('tree_correlation', [0.0, 0.1, 0.2, 0.3]),
                            'min_data_in_leaf': trial.suggest_categorical('min_data_in_leaf', [3, 5, 10, 20]),
                            'max_bin': trial.suggest_categorical('max_bin', [64, 128, 256]),
                            'distribution': trial.suggest_categorical('distribution', ['normal', 'studentt', 'laplace']),
                            'objective': 'mse',
                            'metric': 'rmse',
                            'random_state': 42,
                            'verbose': 0
                        }

                    model = PGBM()
                    model.train((X_train, y_train),
                                objective=mseloss_objective,
                                metric=rmseloss_metric,
                                params=params)

                    y_pred = model.predict(X_test)
                    mse = mean_squared_error(y_test, y_pred)
                    rmse = np.sqrt(mse)

                    rmse_trial_history[pruner_name].append(rmse)

                    # ===== SAVE BEST MODEL =====
                    if rmse < best_rmse_tracker[(model_name, pruner_name)]:
                        best_rmse_tracker[(model_name, pruner_name)] = rmse
                        save_path = os.path.join(
                            model_save_dir,
                            f"{model_name}_{pruner_name}_BEST.pkl"
                        )
                        _ensure_parent_dir(save_path)
                        joblib.dump(model, _ensure_parent_dir(save_path))

                    return mse

                study.optimize(pgbm_objective, n_trials=50)

            else:

                def objective(trial):
                    params = {}
                    for key, values in param_space.items():
                        params[key] = trial.suggest_categorical(key, values)

                    model = model_class(**params)

                    if model_name == 'TabNet':
                        model.fit(X_train, y_train.reshape(-1, 1))
                    else:
                        model.fit(X_train, y_train)


                    y_pred = model.predict(X_test)

                    if model_name == 'TabNet':
                        y_pred = y_pred.ravel()

                    mse = mean_squared_error(y_test, y_pred)

                    rmse = np.sqrt(mse)

                    rmse_trial_history[pruner_name].append(rmse)

                    # ===== SAVE BEST MODEL =====
                    if rmse < best_rmse_tracker[(model_name, pruner_name)]:
                        best_rmse_tracker[(model_name, pruner_name)] = rmse
                        save_path = os.path.join(
                            model_save_dir,
                            f"{model_name}_{pruner_name}_BEST.pkl"
                        )
                        _ensure_parent_dir(save_path)
                        joblib.dump(model, _ensure_parent_dir(save_path))

                    return mse

                study.optimize(objective, n_trials=50)

            elapsed_time = time.time() - start_time

            # Load frozen model (NO RETRAIN)
            best_model = joblib.load(
                os.path.join(model_save_dir, f"{model_name}_{pruner_name}_BEST.pkl")
            )

            y_pred = best_model.predict(X_test)

            if model_name == 'TabNet':
                y_pred = y_pred.ravel()

            mse = mean_squared_error(y_test, y_pred)
            rmse = np.sqrt(mse)
            corr_coef = np.corrcoef(y_test, y_pred)[0, 1]


            predictions_df[f'{model_name}_{pruner_name}_Predicted'] = y_pred

            best_scores[(model_name, pruner_name)] = {
                'best_score': mse,
                'best_params': study.best_params,
                'test_mse': mse,
                'test_rmse': rmse,
                'test_corr_coef': corr_coef,
                'pruner': pruner_name
            }

            timing_records.append({
                'Model': model_name,
                'Pruner': pruner_name,
                'Tuning_Time_Seconds': elapsed_time
            })

        # RMSE plots & Excel writing (UNCHANGED)
        rmse_df = pd.DataFrame(rmse_trial_history)
        rmse_df.insert(0, "Trial", np.arange(1, len(rmse_df) + 1))

        if not os.path.exists(excel_path):
            pd.DataFrame().to_excel(excel_path)
        with pd.ExcelWriter(_ensure_parent_dir(_ensure_excel_file(excel_path)), engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
            writer
            rmse_df.to_excel(writer, sheet_name=f"RMSE_Trials_{model_name}", index=False)
        # ================= SAVE RMSE PLOT =================
        plot_dir = os.path.dirname(excel_path)
        plot_path = os.path.join(plot_dir, f"RMSE_Trials_{model_name}.png")

        plt.figure(figsize=(10, 6))
        for pruner_name, values in rmse_trial_history.items():
            if len(values) > 0:   # <-- important safety check
                plt.plot(values, label=pruner_name, linewidth=2)

        plt.title(f"RMSE Variation Over Trials\n{model_name}")
        plt.xlabel("Trial Number")
        plt.ylabel("RMSE")
        plt.legend(loc="center left", bbox_to_anchor=(1.02, 0.5))
        plt.tight_layout()

        _ensure_parent_dir(plot_path)
        plt.savefig(_ensure_parent_dir(plot_path), dpi=100, bbox_inches="tight")
        plt.close()

        # ================= INSERT PLOT INTO EXCEL =================
        wb = load_workbook(_ensure_excel_file(excel_path))
        ws = wb[f"RMSE_Trials_{model_name}"]

        img = Image(plot_path)
        img.anchor = "J2"
        ws.add_image(img)

        _ensure_parent_dir(excel_path)
        wb.save(_ensure_parent_dir(excel_path))

    timing_df = pd.DataFrame(timing_records)

    if not os.path.exists(excel_path):
        pd.DataFrame().to_excel(excel_path)
    with pd.ExcelWriter(_ensure_parent_dir(_ensure_excel_file(excel_path)), engine='openpyxl', mode='a') as writer:
        writer
        predictions_df.to_excel(writer, sheet_name='Predictions', index=False)
        writer
        timing_df.to_excel(writer, sheet_name='Tuning_Time', index=False)

    return best_scores


best_scores_autosampler = hyperparameter_tuning_all(X_train, y_train, X_test, y_test, "./drive/MyDrive/Regression/pile_settlement_uncertainty_analysis/hyperparameter_tuning/test.xlsx")


[I 2026-05-03 06:28:26,097] A new study created in memory with name: no-name-253f026c-a3cb-4f38-918a-d5d7a9c68189


Running Optuna for Random Forest with MedianPruner...


[I 2026-05-03 06:28:26,913] Trial 0 finished with value: 572.5862117143126 and parameters: {'n_estimators': 300, 'criterion': 'squared_error', 'max_depth': 20, 'min_samples_split': 10, 'min_samples_leaf': 1, 'min_weight_fraction_leaf': 0.2, 'max_features': 0.3, 'max_leaf_nodes': 50, 'min_impurity_decrease': 0.1, 'n_jobs': -1, 'random_state': 42, 'verbose': 0, 'warm_start': False, 'ccp_alpha': 0.001}. Best is trial 0 with value: 572.5862117143126.
[I 2026-05-03 06:28:27,755] Trial 1 finished with value: 272.61050892538816 and parameters: {'n_estimators': 300, 'criterion': 'friedman_mse', 'max_depth': 10, 'min_samples_split': 5, 'min_samples_leaf': 1, 'min_weight_fraction_leaf': 0.01, 'max_features': 0.5, 'max_leaf_nodes': 50, 'min_impurity_decrease': 0.2, 'n_jobs': -1, 'random_state': 42, 'verbose': 0, 'warm_start': False, 'ccp_alpha': 0.05}. Best is trial 1 with value: 272.61050892538816.
[I 2026-05-03 06:28:28,228] Trial 2 finished with value: 511.7081485368545 and parameters: {'n_est

Running Optuna for Random Forest with NopPruner...


[I 2026-05-03 06:29:14,669] Trial 0 finished with value: 305.31770808848694 and parameters: {'n_estimators': 300, 'criterion': 'friedman_mse', 'max_depth': 20, 'min_samples_split': 0.01, 'min_samples_leaf': 0.01, 'min_weight_fraction_leaf': 0.0, 'max_features': 0.5, 'max_leaf_nodes': 200, 'min_impurity_decrease': 0.0, 'n_jobs': -1, 'random_state': 42, 'verbose': 0, 'warm_start': False, 'ccp_alpha': 0.001}. Best is trial 0 with value: 305.31770808848694.
[I 2026-05-03 06:29:15,241] Trial 1 finished with value: 568.1514884757689 and parameters: {'n_estimators': 200, 'criterion': 'poisson', 'max_depth': 10, 'min_samples_split': 5, 'min_samples_leaf': 1, 'min_weight_fraction_leaf': 0.2, 'max_features': 'sqrt', 'max_leaf_nodes': 50, 'min_impurity_decrease': 0.0, 'n_jobs': -1, 'random_state': 42, 'verbose': 0, 'warm_start': False, 'ccp_alpha': 0.05}. Best is trial 0 with value: 305.31770808848694.
[I 2026-05-03 06:29:15,546] Trial 2 finished with value: 312.20875390246925 and parameters: {'n

...

[I 2026-05-03 10:25:30,470] Trial 47 finished with value: 129.13451732896672 and parameters: {'n_estimators': 300, 'learning_rate': 0.1, 'max_leaves': 22, 'min_split_gain': 1.0, 'reg_lambda': 0.1, 'feature_fraction': 0.9, 'bagging_fraction': 0.9, 'tree_correlation': 0.3, 'min_data_in_leaf': 5, 'max_bin': 128, 'distribution': 'studentt'}. Best is trial 32 with value: 110.41650825213384.


Training on CPU


[I 2026-05-03 10:25:40,503] Trial 48 finished with value: 114.53957671038742 and parameters: {'n_estimators': 300, 'learning_rate': 0.1, 'max_leaves': 42, 'min_split_gain': 1.0, 'reg_lambda': 10.0, 'feature_fraction': 0.9, 'bagging_fraction': 0.9, 'tree_correlation': 0.2, 'min_data_in_leaf': 5, 'max_bin': 64, 'distribution': 'normal'}. Best is trial 32 with value: 110.41650825213384.


Training on CPU


[I 2026-05-03 10:25:49,085] Trial 49 finished with value: 114.53957671038742 and parameters: {'n_estimators': 300, 'learning_rate': 0.1, 'max_leaves': 46, 'min_split_gain': 1.0, 'reg_lambda': 10.0, 'feature_fraction': 0.9, 'bagging_fraction': 0.9, 'tree_correlation': 0.2, 'min_data_in_leaf': 5, 'max_bin': 64, 'distribution': 'normal'}. Best is trial 32 with value: 110.41650825213384.


In [12]:
best_scores_autosampler

{('Random Forest', 'MedianPruner'): {'best_score': 13.182650929420474,
  'best_params': {'n_estimators': 700,
   'criterion': 'friedman_mse',
   'max_depth': None,
   'min_samples_split': 5,
   'min_samples_leaf': 1,
   'min_weight_fraction_leaf': 0.0,
   'max_features': 'sqrt',
   'max_leaf_nodes': 100,
   'min_impurity_decrease': 0.0,
   'n_jobs': -1,
   'random_state': 42,
   'verbose': 0,
   'warm_start': False,
   'ccp_alpha': 0.001},
  'test_mse': 13.182650929420474,
  'test_rmse': 3.63079205262715,
  'test_corr_coef': 0.9362480867949565,
  'pruner': 'MedianPruner'},
 ('Random Forest', 'NopPruner'): {'best_score': 11.272303056476819,
  'best_params': {'n_estimators': 100,
   'criterion': 'friedman_mse',
   'max_depth': 30,
   'min_samples_split': 2,
   'min_samples_leaf': 1,
   'min_weight_fraction_leaf': 0.0,
   'max_features': 0.5,
   'max_leaf_nodes': 50,
   'min_impurity_decrease': 0.1,
   'n_jobs': -1,
   'random_state': 42,
   'verbose': 0,
   'warm_start': False,
   'ccp_a

# **Best Model Analysis**

In [13]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import os
def get_best_models_and_predict(best_scores_autosampler, X_train, y_train, X_test, y_test, file_path):
    # Convert input data to NumPy arrays
    X_train = np.array(X_train)
    y_train = np.array(y_train)
    X_test = np.array(X_test)
    y_test = np.array(y_test)

    # Mapping for model creation based on dictionary keys
    model_mapping = {
        'Random Forest': RandomForestRegressor,
        'Gradient Boosting': GradientBoostingRegressor,
        'XGBoost': XGBRegressor,
        'LightGBM': LGBMRegressor,
        'CatBoost': CatBoostRegressor,
        'GPBoost': GPBoostRegressor,
        'NGBoost': NGBRegressor,
        'TabNet': TabNetRegressor,
        'HistGradientBoosting': HistGradientBoostingRegressor,
        'PGBM': PGBM  # PGBM is handled separately
    }

    # Dictionary to store the best model for each type
    best_models = {}

    # Iterate over the dictionary to find the best pruner for each model type
    for (model_name, pruner), params in best_scores_autosampler.items():
        current_score = params.get('test_mse', np.inf)
        if model_name not in best_models or current_score < best_models[model_name]['score']:
            best_models[model_name] = {
                'score': current_score,
                'params': params['best_params'],
                'pruner': pruner
            }

    # Prepare a DataFrame to store predictions
    df = pd.read_csv(file_path)

    # Iterate over the best models to train and predict
    for model_name, model_info in best_models.items():
        best_params = model_info['params']
        model_class = model_mapping.get(model_name)

        if model_class is None:
            print(f"Model {model_name} is not supported or not available.")
            continue

        # Handle specific parameters or settings for model if needed
        if model_name == 'CatBoost':
            best_params.pop('verbose', None)  # Remove 'verbose' for CatBoost

        # Create an instance of the best model with the best parameters
        if model_name == 'PGBM':
            model = model_class()
            model.train((X_train, y_train), objective=mseloss_objective, metric=rmseloss_metric, params=best_params)
            predictions = model.predict(X_test)
        elif model_name == 'TabNet':
            model = model_class(**best_params)
            model.fit(X_train, y_train.reshape(-1, 1))
            predictions = model.predict(X_test)
            predictions = predictions.ravel()
        else:
            model = model_class(**best_params)
            model.fit(X_train, y_train)
            predictions = model.predict(X_test)

        # Add predictions to the DataFrame
        df[f'{model_name} Predictions'] = predictions

        # Plot actual vs. predicted
        plt.figure(figsize=(10, 6))
        plt.scatter(y_test, predictions, alpha=0.6)
        plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'k--', color='red', lw=2)
        plt.xlabel("Actual str")
        plt.ylabel("Predicted str")
        plt.title(f"Actual vs. Predicted Values ({model_name})")
        plt.grid(True)
        plt.tight_layout()

        # Save the plot temporarily
        plot_path = f'temp_plot_{model_name}.png'
        _ensure_parent_dir(plot_path)
        plt.savefig(_ensure_parent_dir(plot_path))
        plt.close()

    # ✅ NEW OUTPUT DIRECTORY
    output_dir = "./drive/MyDrive/Regression/pile_settlement_uncertainty_analysis/hyperparameter_tuning/"
    os.makedirs(output_dir, exist_ok=True)

    output_excel_path = os.path.join(
        output_dir,
        os.path.basename(file_path).replace('.csv', '_results.xlsx')
    )

    # Save predictions and plots to Excel
    with pd.ExcelWriter(_ensure_parent_dir(output_excel_path), engine='xlsxwriter') as writer:
        # Write data to Excel
        writer
        df.to_excel(writer, sheet_name='data', index=False)

        # Get the xlsxwriter objects
        workbook = writer.book

        # Insert each plot into a separate worksheet
        for model_name in best_models.keys():
            short_model_name = ''.join([word[0] for word in model_name.split()])
            sheet_name = f'{short_model_name}_Plot'

            worksheet = workbook.add_worksheet(sheet_name)
            writer.sheets[sheet_name] = worksheet
            plot_path = f'temp_plot_{model_name}.png'
            worksheet.insert_image('A1', plot_path)

    # Clean up temporary plot files
    for model_name in best_models.keys():
        if os.path.exists(str(f'temp_plot_{model_name}.png')): os.remove(str(f'temp_plot_{model_name}.png'))

    return df, best_models

# Call the function
df, best_models = get_best_models_and_predict(best_scores_autosampler, X_train, y_train, X_test, y_test, "./drive/MyDrive/Regression/pile_settlement_uncertainty_analysis/data/test.csv")

0:	learn: 9.5347619	total: 9.54ms	remaining: 9.54s
1:	learn: 9.0999310	total: 10.7ms	remaining: 5.35s
2:	learn: 8.6970867	total: 11.2ms	remaining: 3.73s
3:	learn: 8.3330568	total: 12.3ms	remaining: 3.07s
4:	learn: 7.9749522	total: 12.7ms	remaining: 2.54s
5:	learn: 7.7317051	total: 13.3ms	remaining: 2.2s
6:	learn: 7.5073418	total: 13.7ms	remaining: 1.94s
7:	learn: 7.2145204	total: 13.9ms	remaining: 1.73s
8:	learn: 7.0309963	total: 14.3ms	remaining: 1.58s
9:	learn: 6.7941692	total: 14.6ms	remaining: 1.45s
10:	learn: 6.6639794	total: 14.8ms	remaining: 1.33s
11:	learn: 6.5511978	total: 15.2ms	remaining: 1.25s
12:	learn: 6.3588168	total: 15.4ms	remaining: 1.17s
13:	learn: 6.2149613	total: 15.5ms	remaining: 1.09s
14:	learn: 6.0902266	total: 15.7ms	remaining: 1.03s
15:	learn: 5.9715747	total: 15.9ms	remaining: 977ms
16:	learn: 5.8326751	total: 16.1ms	remaining: 932ms
17:	learn: 5.6879740	total: 16.4ms	remaining: 892ms
18:	learn: 5.6187002	total: 16.6ms	remaining: 856ms
19:	learn: 5.5260604	to

In [14]:
plot_best_scores(best_scores_autosampler,"./drive/MyDrive/Regression/pile_settlement_uncertainty_analysis/hyperparameter_tuning/test_results.xlsx")

In [15]:
generate_interpretml_explanations_summary_pruners(best_scores_autosampler, X_train, y_train, x_test, feature_names,excel_file_path = "./drive/MyDrive/Regression/pile_settlement_uncertainty_analysis/hyperparameter_tuning/test_results.xlsx")

  0%|          | 0/95 [00:00<?, ?it/s]

  0%|          | 0/95 [00:00<?, ?it/s]

  0%|          | 0/95 [00:00<?, ?it/s]

  0%|          | 0/95 [00:00<?, ?it/s]

  0%|          | 0/95 [00:00<?, ?it/s]

...

  0%|          | 0/95 [00:00<?, ?it/s]

epoch 0  | loss: 1863.62427| val_0_mse: 22337.47266|  0:00:00s
epoch 1  | loss: 1343.00208| val_0_mse: 3917.91699|  0:00:00s
epoch 2  | loss: 990.86151| val_0_mse: 4074.80615|  0:00:00s
epoch 3  | loss: 710.26379| val_0_mse: 6950.6084|  0:00:00s
epoch 4  | loss: 488.672 | val_0_mse: 10653.85938|  0:00:01s
epoch 5  | loss: 381.52161| val_0_mse: 10908.01953|  0:00:01s
epoch 6  | loss: 260.25192| val_0_mse: 12309.05664|  0:00:01s
epoch 7  | loss: 173.44859| val_0_mse: 12887.03711|  0:00:01s
epoch 8  | loss: 108.27684| val_0_mse: 11605.8916|  0:00:02s
epoch 9  | loss: 81.52638| val_0_mse: 7372.39307|  0:00:02s
epoch 10 | loss: 65.07398| val_0_mse: 6870.8999|  0:00:02s
epoch 11 | loss: 77.02025| val_0_mse: 5492.7124|  0:00:03s

Early stopping occurred at epoch 11 with best_epoch = 1 and best_val_0_mse = 3917.91699


  0%|          | 0/95 [00:00<?, ?it/s]

  0%|          | 0/95 [00:00<?, ?it/s]

Model PGBM is not supported or not available.
